In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
# Argumentos de estabilidad para Docker
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

try:
    # El Manager instala el driver compatible con la versión de Chrome instalada
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    driver.get("https://www.google.com")
    print(f"¡CONECTADO! Título de la página: {driver.title}")
    driver.quit()

except Exception as e:
    print(f"Error: {e}")

¡CONECTADO! Título de la página: Google


In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN (Configuración de Laboratorio)
# ==========================================================
def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    )
    
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

# ==========================================================
# 2. CONFIGURACIÓN DEL GRUPO (Cada grupo modifica aquí)
# ==========================================================

# Pegar la URL del sitio asignado (Retail, Energía, E-commerce, etc.)
URL_OBJETIVO = "https://es.investing.com/commodities/"

# Definir las etiquetas técnicas (clases CSS) encontradas con el Inspector de Chrome
SELECTOR_CONTENEDOR = "tr[data-pair-id]"  # Fila de cada commodity
SELECTOR_DATO_A = "td.first.elp.popper-trigger"  # Nombre
SELECTOR_DATO_B = "td.bold[data-value]"  # Precio

# ==========================================================
# 3. EJECUCIÓN DEL SCRAPING
# ==========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print("Conectando a la fuente de datos...")
    driver.get(URL_OBJETIVO)
    time.sleep(5)  # Tiempo de espera para carga de datos dinámicos

    # Capturamos todos los bloques de información
    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f"Se encontraron {len(elementos)} registros potenciales.")

    for item in elementos:
        try:
            # Extracción genérica de datos
            columna_a = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
            columna_b = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text

            # Limpieza básica de caracteres no numéricos (opcional según el proyecto)
            valor_limpio = "".join(filter(str.isdigit, columna_b))

            dataset_final.append({
                "variable_nombre": columna_a,
                "variable_valor": valor_limpio,
                "status": 1.0  # Indicador de registro procesado (float)
            })
        except:
            # Si un elemento está vacío o es un anuncio, saltar al siguiente
            continue

# ==========================================================
# 4. SALIDA DE DATOS (Para análisis posterior)
# ==========================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "datos_extraidos_grupo.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f"Proceso exitoso. Archivo generado: {nombre_archivo}")
        print(df.head())  # Muestra de los primeros 5 datos
    else:
        print("No se capturaron datos. Revisar los selectores CSS.")

finally:
    driver.quit()
    print("Navegador cerrado.")

In [5]:
# ============================================================
# BLOQUE PARA JUPYTER NOTEBOOK - Web Scraping Commodities
# VERSIÓN CORREGIDA CON SELECTORES ESPECÍFICOS
# ============================================================

import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# =========================================================
# 1. MOTOR DE NAVEGACIÓN
# =========================================================
def iniciar_navegador():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    )
    
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

# =========================================================
# 2. CONFIGURACIÓN CON SELECTORES CORREGIDOS
# =========================================================
URL_OBJETIVO = "https://es.investing.com/commodities/"

# Selectores basados en las clases encontradas
SELECTOR_TABLA = "table.datatable-v2_table__93S4Y"
SELECTOR_FILAS = "tr.datatable-v2_row__hkEus"
SELECTOR_CELDAS = "td"

# =========================================================
# 3. EJECUCIÓN DEL SCRAPING
# =========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print("🌾 Conectando a la fuente de datos (Investing.com)...")
    driver.get(URL_OBJETIVO)
    
    print("⏳ Esperando carga de tabla dinámica...")
    
    # Espera a que cargue la tabla
    tabla = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, SELECTOR_TABLA))
    )
    
    time.sleep(5)
    
    # Busca todas las filas
    filas = driver.find_elements(By.CSS_SELECTOR, SELECTOR_FILAS)
    print(f"✓ Se encontraron {len(filas)} commodities\n")
    
    if len(filas) == 0:
        print("⚠️ No se encontraron filas. Intentando selector alternativo...")
        # Intenta con selector genérico
        filas = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
        print(f"✓ Con selector genérico: {len(filas)} filas encontradas")
    
    for indice, fila in enumerate(filas, 1):
        try:
            celdas = fila.find_elements(By.TAG_NAME, "td")
            
            if len(celdas) >= 2:
                nombre = celdas[0].text.strip()
                precio = celdas[1].text.strip()
                cambio = celdas[2].text.strip() if len(celdas) > 2 else "N/A"
                
                precio_limpio = "".join(
                    filter(lambda x: x.isdigit() or x == ".", precio)
                )
                
                if nombre and precio_limpio:
                    dataset_final.append({
                        "commodity": nombre,
                        "precio": precio_limpio,
                        "cambio": cambio,
                        "precio_original": precio,
                        "status": 1.0
                    })
                    
                    print(f"  ✓ {nombre}: {precio} ({cambio})")
        
        except Exception as e:
            continue

    print("\n" + "="*70)
    
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "datos_commodities_investing.csv"
        df.to_csv(nombre_archivo, index=False)
        
        print(f"✓ Éxito! {len(dataset_final)} registros capturados")
        print(f"✓ Archivo: {nombre_archivo}")
        print(f"\n📊 Datos:")
        print(df.head(10))
    
    else:
        print("⚠ No se capturaron datos")
        print("Verifica los selectores CSS en el inspector")

except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

finally:
    driver.quit()
    print("\n✓ Navegador cerrado")

🌾 Conectando a la fuente de datos (Investing.com)...
❌ Error: HTTPConnectionPool(host='localhost', port=46235): Read timed out. (read timeout=120)


Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/urllib3/connection.py", line 571, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/http/client.py", line 1378, in getresponse
    response.begin()
  File "/opt/conda/lib/python3.11/http/client.py", line 318, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/http/client.py", line 279, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
TimeoutError: tim


✓ Navegador cerrado
